# Práctica A: ¿Puede la IA predecir el clima? Benchmark Punchana (SENAMHI Iquitos)

Este notebook implementa el **Caso 1** de la investigación:
- Evaluación rigurosa sobre **2,557 días** reales de la estación meteorológica Punchana (Loreto, Perú, 91 msnm).
- Split temporal ciego de **90 días de holdout** (Octubre a Diciembre 2025).
- **Benchmark ampliado de 14 modelos** abarcando todas las familias teóricas:
  1. Baselines (`Naive 1d`, `Naive 7d`)
  2. Econométricos (`ARIMA`, `SARIMA`, `Holt-Winters` amortiguado)
  3. Machine Learning Tabular (`Ridge`, `Random Forest`, `ExtraTrees`, `LightGBM`)
  4. Deep Learning Local (`PyTorch LSTM` con ensamble de 5 semillas aleatorias)
  5. Foundation Models Zero-Shot (`Chronos-Bolt` Amazon, `TimesFM 2.5` Google)
  6. **Física Global de Supercómputo** (`ECMWF IFS` ~9 km de Europa y `NOAA GFS` ~28 km de EE.UU.)
- Evaluación con **Prueba de Diebold-Mariano (HLN 1997)**, cobertura de incertidumbre empírica P10–P90 y regla de parsimonia.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Cargar benchmark consolidado de los 14 modelos
df_bm = pd.read_csv('../data/caso_1/benchmark_punchana.csv')
print('=== BENCHMARK DE 14 MODELOS (HOLDOUT CIEGO 90 DÍAS) ===')
display(df_bm[['modelo', 'familia', 'rmse_1d', 'mae_1d', 'rmse_7d', 'tiempo_s', 'hardware', 'p_valor_vs_mejor']])

## 2. Visualización Comparativa del Benchmark (14 Modelos)
Comparamos el error RMSE a 1 día. Notemos cómo los modelos locales de ML (`ExtraTrees`, `Random Forest`) y los foundation models (`TimesFM`, `Chronos`) superan a la física global de supercómputo (`ECMWF IFS` y `NOAA GFS`), y cómo `ARIMA (2,0,1)` empata estadísticamente en precisión con los modelos más pesados.

In [ ]:
df_sorted = df_bm.sort_values('rmse_1d', ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 7))
colors = []
for _, row in df_sorted.iterrows():
    if 'Física Global' in row['familia']:
        colors.append('#8b5cf6')
    elif row['p_valor_vs_mejor'] == 1.0:
        colors.append('#059669')
    elif row['p_valor_vs_mejor'] > 0.05:
        colors.append('#10b981')
    elif 'Baseline' in row['familia']:
        colors.append('#ef4444')
    else:
        colors.append('#f59e0b')

bars = ax.barh(df_sorted['modelo'], df_sorted['rmse_1d'], color=colors, height=0.65, edgecolor='#334155')
ax.set_xlabel('RMSE a 1 Día (°C) en Holdout 90 días (Menor es mejor)', fontweight='bold')
ax.set_title('Benchmark de 14 Modelos: Física Global vs ML Local vs Foundation Models', fontweight='bold', pad=12)
ax.grid(axis='x', linestyle='--', alpha=0.5)
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.03, bar.get_y() + bar.get_height()/2, f'{width:.2f} °C', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Confrontación: Física Global de Supercómputo vs Estación de Superficie SENAMHI
Evaluamos el comportamiento de los modelos globales numéricos (NWP) de la **NOAA (GFS, malla de ~28 km)** y del **ECMWF (IFS, malla de ~9 km)** comparados directamente contra los termómetros en tierra de SENAMHI Punchana.

In [ ]:
df_g = pd.read_csv('../data/caso_1/noaa_ecmwf_forecast.csv', parse_dates=['fecha'])
df_s = pd.read_csv('../data/caso_1/senamhi_diario.csv', parse_dates=['fecha'])
df_merged = pd.merge(df_g, df_s[['fecha', 'tmax']], on='fecha', how='inner').rename(columns={'tmax': 'tmax_senamhi'})

mae_ecmwf = (df_merged['tmax_ecmwf'] - df_merged['tmax_senamhi']).abs().mean()
mae_gfs = (df_merged['tmax_noaa_gfs'] - df_merged['tmax_senamhi']).abs().mean()
rmse_ecmwf = np.sqrt(((df_merged['tmax_ecmwf'] - df_merged['tmax_senamhi'])**2).mean())
rmse_gfs = np.sqrt(((df_merged['tmax_noaa_gfs'] - df_merged['tmax_senamhi'])**2).mean())
dif_inter = (df_merged['tmax_noaa_gfs'] - df_merged['tmax_ecmwf']).abs().mean()

print(f'ECMWF IFS: MAE = {mae_ecmwf:.2f} °C | RMSE = {rmse_ecmwf:.2f} °C')
print(f'NOAA GFS:  MAE = {mae_gfs:.2f} °C | RMSE = {rmse_gfs:.2f} °C')
print(f'Discrepancia promedio entre NOAA y ECMWF: {dif_inter:.2f} °C')

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(df_merged['fecha'], df_merged['tmax_senamhi'], 'k-', label='SENAMHI (Observado en Tierra)', linewidth=1.8)
ax.plot(df_merged['fecha'], df_merged['tmax_ecmwf'], color='#8b5cf6', linestyle='--', label=f'ECMWF IFS (RMSE={rmse_ecmwf:.2f}°C)', linewidth=1.4)
ax.plot(df_merged['fecha'], df_merged['tmax_noaa_gfs'], color='#f97316', linestyle=':', label=f'NOAA GFS (RMSE={rmse_gfs:.2f}°C)', linewidth=1.4)
ax.set_ylabel('Temperatura Máxima (°C)', fontweight='bold')
ax.set_title('Supercómputo Global vs Observación en Tierra en Punchana (Iquitos)', fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Pronóstico Multi-Modelo Operativo a 7 Días y Cono de Incertidumbre P10/P90

In [ ]:
df_fc = pd.read_csv('../data/caso_1/senamhi_forecast_7d.csv')
print('Pronóstico a 7 días:')
display(df_fc)

In [ ]:
plt.figure(figsize=(11, 5.5))
x = range(len(df_fc))
plt.plot(df_fc['fecha'], df_fc['tmax_observada'], 'k-o', label='Observado Real (SENAMHI)', linewidth=2.5)
plt.plot(df_fc['fecha'], df_fc['pred_extratrees'], 'g--s', label='ExtraTrees (Ganador CPU)', linewidth=2)
plt.plot(df_fc['fecha'], df_fc['pred_timesfm'], 'b--^', label='TimesFM 2.5 Zero-Shot (Google)', linewidth=1.8)
plt.plot(df_fc['fecha'], df_fc['pred_ecmwf'], color='#8b5cf6', linestyle='-.', label='ECMWF IFS (Europa 9km)', linewidth=1.5)
plt.plot(df_fc['fecha'], df_fc['pred_noaa_gfs'], color='#f97316', linestyle=':', label='NOAA GFS (EE.UU. 28km)', linewidth=1.5)
plt.fill_between(df_fc['fecha'], df_fc['ci_lower_p10'], df_fc['ci_upper_p90'], color='green', alpha=0.15, label='Banda ExtraTrees P10-P90')
plt.title('Pronóstico Multi-Modelo Últimos 7 Días vs Física Global y Verdad Terreno', fontweight='bold')
plt.ylabel('Temperatura Máxima (°C)', fontweight='bold')
plt.xticks(rotation=45)
plt.legend(ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()